# Enterprise AI Meeting Assistant — Evaluation Engine

Evaluation notebook for block **5. Storage & Evaluation Subsystem** of the system architecture.

| Part | What is evaluated | Metrics |
|---|---|---|
| **A. Representation models** | AE, VAE, GAN, Diffusion | MSE, PSNR, SSIM, FID, diversity |
| **B. ASR** | Whisper transcripts | WER, CER |
| **C. NLP** | FLAN-T5 outputs | ROUGE-1/2/L, BERTScore, Precision/Recall/F1 |

**Before running**
1. Place this notebook in the same folder as `preprocessing.py`, `autoencoder.py`, `vae.py`, `gan.py`, `diffusion.py` and `transformer_pipeline.py`.
2. `data/X_train.pt`, `X_val.pt`, `X_test.pt` must exist (run `preprocessing.py`).
3. Train the models you want compared. Any missing checkpoint in `checkpoints/` is skipped automatically:
   `autoencoder.pt`, `vae.pt`, `gan.pt`, `diffusion.pt`.

## 0. Setup

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())          # notebook must sit next to the .py files

from preprocessing import load_splits, N_MELS, N_FRAMES
from autoencoder import Autoencoder, MelSpectrogramDataset, simple_ssim, psnr
from vae import VAE, evaluate_vae
from gan import GAN, load_feature_extractor, extract_features, frechet_distance, mean_pairwise_l1
from diffusion import UNet, Diffusion, generate as diffusion_generate, refine as diffusion_refine

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR, OUT_DIR = "checkpoints", "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- evaluation settings ----
N_GEN       = 500    # generated samples per model (for FID / diversity)
DDIM_STEPS  = 50     # diffusion sampling steps (iterative refinement)
REFINE_T    = 250    # noise level for the diffusion denoise-and-compare test
SEED        = 42

torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE)

## 1. Load data

In [ ]:
X_train, X_val, X_test = load_splits()
N_GEN = min(N_GEN, len(X_test))
print(f"train {tuple(X_train.shape)} | val {tuple(X_val.shape)} | test {tuple(X_test.shape)}")
print(f"value range: [{X_test.min():.3f}, {X_test.max():.3f}]")

## 2. Load trained checkpoints

In [ ]:
def load_ckpt(name):
    path = os.path.join(CKPT_DIR, f"{name}.pt")
    if not os.path.exists(path):
        print(f"[SKIP] {path} not found")
        return None
    return torch.load(path, map_location=DEVICE)

models, meta = {}, {}

ck = load_ckpt("autoencoder")
if ck:
    m = Autoencoder(latent_dim=ck["latent_dim"], dropout=ck["dropout"]).to(DEVICE)
    m.load_state_dict(ck["model_state"]); models["AE"] = m.eval()
    meta["AE"] = {"latent_dim": ck["latent_dim"]}

ck = load_ckpt("vae")
if ck:
    m = VAE(latent_dim=ck["latent_dim"], dropout=ck["dropout"]).to(DEVICE)
    m.load_state_dict(ck["model_state"]); models["VAE"] = m.eval()
    meta["VAE"] = {"latent_dim": ck["latent_dim"], "beta": ck["beta"]}

ck = load_ckpt("gan")
if ck:
    m = GAN(latent_dim=ck["latent_dim"], dropout=ck["dropout"]).to(DEVICE)
    m.load_state_dict(ck["model_state"]); models["GAN"] = m.eval()
    meta["GAN"] = {"latent_dim": ck["latent_dim"]}

diffusion = None
ck = load_ckpt("diffusion")
if ck:
    m = UNet(base=ck["base_channels"], dropout=ck["dropout"]).to(DEVICE)
    m.load_state_dict(ck["model_state"]); models["Diffusion"] = m.eval()
    diffusion = Diffusion(ck["timesteps"])
    meta["Diffusion"] = {"timesteps": ck["timesteps"], "base_channels": ck["base_channels"]}

print("Loaded:", list(models))
for k, v in meta.items():
    print(f"  {k}: {v}")

**FID feature extractor.** FID is computed on features from the trained Autoencoder's encoder
(the standard Inception network is built for natural RGB photos, not Mel spectrograms).
Caveat: this slightly favours the AE itself, so read the AE's FID as an optimistic reference.

In [ ]:
encoder = load_feature_extractor()
real_feats = extract_features(encoder, X_test[:N_GEN])

# Reference: how far apart are two sets of REAL data? (lower bound for any model's FID)
fid_floor = frechet_distance(extract_features(encoder, X_train[:N_GEN]), real_feats)
print(f"FID floor (real train vs real test): {fid_floor:.3f}")

## 3. Part A — Representation models

### 3.1 Reconstruction quality (AE, VAE, Diffusion)
- **AE**: encode → decode.
- **VAE**: encode → decode using the posterior mean μ (deterministic, no sampling noise).
- **Diffusion**: noise each spectrogram to level `REFINE_T`, then denoise it back (iterative refinement).
- **GAN**: has no encoder, so it has no reconstruction metrics (shown as N/A).

In [ ]:
@torch.no_grad()
def batched(fn, X, bs=64):
    return torch.cat([fn(X[i:i + bs].to(DEVICE)).cpu() for i in range(0, len(X), bs)])

recon_fns = {}
if "AE" in models:
    recon_fns["AE"] = lambda x: models["AE"](x)[0]
if "VAE" in models:
    recon_fns["VAE"] = lambda x: models["VAE"].decoder(models["VAE"].encoder(x)[0])
if "Diffusion" in models:
    recon_fns["Diffusion"] = lambda x: diffusion_refine(models["Diffusion"], diffusion, x, REFINE_T, DDIM_STEPS)

def recon_metrics(X_hat, X):
    mse = nn.functional.mse_loss(X_hat, X).item()
    return {
        "MSE": mse,
        "MAE": nn.functional.l1_loss(X_hat, X).item(),
        "PSNR (dB)": psnr(mse),
        "SSIM": simple_ssim(X, X_hat),
    }

recons, recon_results = {}, {}
for name, fn in recon_fns.items():
    print(f"Reconstructing test set with {name}...")
    recons[name] = batched(fn, X_test)
    recon_results[name] = recon_metrics(recons[name], X_test)

pd.DataFrame(recon_results).T.round(5)

### 3.2 Generative quality (all four models)
Each model generates `N_GEN` new spectrograms from scratch:
- **AE** has no generative prior, so a Gaussian is fitted to the training-set latent codes and sampled.
- **VAE**: z ~ N(0, I) → decoder.
- **GAN**: z ~ N(0, I) → generator.
- **Diffusion**: pure noise → DDIM iterative refinement.

In [ ]:
gen_fns = {}

if "AE" in models:
    with torch.no_grad():
        Z = torch.cat([models["AE"].encoder(X_train[i:i + 64].to(DEVICE))[0].cpu()
                       for i in range(0, len(X_train), 64)]).numpy()
    mu_z = Z.mean(0)
    cov_z = np.cov(Z, rowvar=False) + 1e-4 * np.eye(Z.shape[1])

    @torch.no_grad()
    def sample_ae(n):
        z = torch.from_numpy(np.random.multivariate_normal(mu_z, cov_z, n)).float().to(DEVICE)
        return models["AE"].decoder(z).cpu()
    gen_fns["AE"] = sample_ae

if "VAE" in models:
    gen_fns["VAE"] = lambda n: models["VAE"].sample(n).cpu()
if "GAN" in models:
    gen_fns["GAN"] = lambda n: models["GAN"].sample(n).cpu()
if "Diffusion" in models:
    gen_fns["Diffusion"] = lambda n: diffusion_generate(models["Diffusion"], diffusion, n, DDIM_STEPS).cpu()

samples, gen_results = {}, {}
div_real = mean_pairwise_l1(X_test[:N_GEN])
for name, fn in gen_fns.items():
    print(f"Generating {N_GEN} samples with {name}...")
    samples[name] = fn(N_GEN)
    gen_results[name] = {
        "FID": frechet_distance(real_feats, extract_features(encoder, samples[name])),
        "Diversity ratio": mean_pairwise_l1(samples[name]) / div_real,
        "Sample mean": samples[name].mean().item(),
    }

print(f"Real test-set mean: {X_test.mean():.4f} | FID floor: {fid_floor:.3f}")
pd.DataFrame(gen_results).T.round(4)

**Reading the generative table**
- **FID**: lower is better; compare against the real-vs-real floor above.
- **Diversity ratio**: generated-sample variety divided by real-data variety. Near 1.0 is healthy; well below 1 means mode collapse (most common in GANs).

### 3.3 Combined comparison table

In [ ]:
order = [n for n in ["AE", "VAE", "GAN", "Diffusion"] if n in models]
results = pd.DataFrame(index=order, columns=["MSE", "MAE", "PSNR (dB)", "SSIM", "FID", "Diversity ratio"], dtype=float)
for n in order:
    for k, v in recon_results.get(n, {}).items():
        results.loc[n, k] = v
    for k in ["FID", "Diversity ratio"]:
        if n in gen_results:
            results.loc[n, k] = gen_results[n][k]

results.to_csv(os.path.join(OUT_DIR, "evaluation_results.csv"))
results.round(4).fillna("N/A")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (col, better) in zip(axes, [("PSNR (dB)", "higher"), ("SSIM", "higher"),
                                     ("FID", "lower"), ("Diversity ratio", "closer to 1")]):
    vals = results[col].astype(float)
    ax.bar(vals.index, vals.fillna(0).values, color=["#5b8def", "#9b6bff", "#2fbf9b", "#f2994a"][:len(vals)])
    ax.set_title(f"{col}\n({better} is better)", fontsize=10)
    for i, v in enumerate(vals.values):
        ax.text(i, 0 if np.isnan(v) else v, "N/A" if np.isnan(v) else f"{v:.3g}", ha="center", va="bottom", fontsize=9)
    if col == "Diversity ratio":
        ax.axhline(1.0, ls="--", c="gray", lw=1)
    if col == "FID":
        ax.axhline(fid_floor, ls="--", c="gray", lw=1)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "model_comparison.png"), dpi=130)
plt.show()

### 3.4 Visual inspection

In [ ]:
def show_rows(rows, n=6, title=None, path=None):
    fig, axes = plt.subplots(len(rows), n, figsize=(2.2 * n, 1.9 * len(rows)), squeeze=False)
    for r, (name, tensor) in enumerate(rows.items()):
        for i in range(n):
            axes[r, i].imshow(tensor[i, 0].numpy(), origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=1)
            axes[r, i].axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=10)
    if title:
        fig.suptitle(title, y=1.0)
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=130)
    plt.show()

print("Reconstructions (same test spectrograms):")
show_rows({"Real": X_test, **recons}, title="Reconstruction comparison",
          path=os.path.join(OUT_DIR, "reconstructions.png"))

In [ ]:
print("Generated samples (new spectrograms):")
show_rows({"Real": X_test, **samples}, title="Generated samples",
          path=os.path.join(OUT_DIR, "generated_samples.png"))

In [ ]:
# Frequency profile: does each model reproduce the real average energy per Mel bin?
plt.figure(figsize=(8, 3.6))
plt.plot(X_test[:N_GEN].mean(dim=(0, 1, 3)).numpy(), "k", lw=2.5, label="Real")
for name, s in samples.items():
    plt.plot(s.mean(dim=(0, 1, 3)).numpy(), lw=1.5, label=name)
plt.xlabel("Mel bin"); plt.ylabel("Mean normalized energy"); plt.legend(); plt.grid(alpha=.3)
plt.title("Average spectral profile: real vs generated")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "spectral_profile.png"), dpi=130)
plt.show()

### 3.5 Model-specific diagnostics

In [ ]:
if "VAE" in models:
    import vae as vae_module
    loader = DataLoader(MelSpectrogramDataset(X_test), batch_size=16)
    vm = evaluate_vae(models["VAE"], loader, beta=meta["VAE"]["beta"])
    print("VAE test metrics (beta = %.3g):" % meta["VAE"]["beta"])
    for k, v in vm.items():
        print(f"  {k}: {v:.5f}")
    print("\nRule of thumb: ~0.1-0.5 nats/dim = well-regularized latent space; several nats/dim = noisy random samples.")

In [ ]:
if "GAN" in models:
    gan = models["GAN"]
    with torch.no_grad():
        d_real = torch.cat([gan.discriminator(X_test[i:i + 64].to(DEVICE)).cpu() for i in range(0, len(X_test), 64)])
        d_fake = torch.cat([gan.discriminator(samples["GAN"][i:i + 64].to(DEVICE)).cpu()
                            for i in range(0, len(samples["GAN"]), 64)])
    print(f"Discriminator accuracy on REAL test data : {(d_real > 0).float().mean():.3f}")
    print(f"Discriminator accuracy on GENERATED data : {(d_fake <= 0).float().mean():.3f}")
    print("Healthy balance is roughly 0.5-0.8 on both. Near 1.0/1.0 means D has beaten G.")

In [ ]:
if "Diffusion" in models:
    # Reconstruction error as a function of how much noise is added (harder = larger t)
    ts = [50, 100, 250, 500, 750]
    sub = X_test[:128]
    rows = []
    for t in ts:
        rec = batched(lambda x: diffusion_refine(models["Diffusion"], diffusion, x, t, DDIM_STEPS), sub)
        rows.append({"noise level t": t, **recon_metrics(rec, sub)})
    diff_curve = pd.DataFrame(rows).set_index("noise level t")
    display(diff_curve.round(5))
    diff_curve["MSE"].plot(marker="o", title="Diffusion refinement MSE vs. starting noise level", figsize=(6, 3))
    plt.ylabel("MSE"); plt.grid(alpha=.3); plt.show()

## 4. Parts B & C — ASR and NLP evaluation

Uses `transformer_pipeline.py` (Whisper → FLAN-T5). Set `AUDIO_PATH` and the reference
texts below to evaluate a real meeting; leave `AUDIO_PATH = None` to run a small demo of the
metrics on toy strings.

Extra packages: `pip install rouge-score bert-score`.

In [ ]:
from transformer_pipeline import (word_error_rate, character_error_rate,
                                  evaluate_summary, precision_recall_f1)

# ---- set these to evaluate a real meeting ----
AUDIO_PATH      = None            # e.g. "meeting.wav"
REF_TRANSCRIPT  = ""              # human ground-truth transcript
REF_SUMMARY     = ""              # human reference summary
REF_DECISIONS   = []              # ground-truth decisions (list of strings)
REF_ACTIONS     = []              # ground-truth action items, e.g. "Priya -> send report -> Friday"

In [ ]:
if AUDIO_PATH:
    from transformer_pipeline import run_end_to_end
    out = run_end_to_end(AUDIO_PATH)
    hyp_transcript, hyp_summary = out["transcript"], out["summary"]
    hyp_decisions, hyp_actions = out["decisions"], out["action_items"]
    print("Ran full pipeline on", AUDIO_PATH)
else:
    print("AUDIO_PATH not set -> running metric demo on toy examples.")
    REF_TRANSCRIPT = "we decided to launch the product next friday and priya will send the report"
    hyp_transcript = "we decide to launch the product next friday and priya will send report"
    REF_SUMMARY = "The team agreed to launch next Friday and Priya will send the report."
    hyp_summary = "The team will launch the product on Friday; Priya sends the report."
    REF_DECISIONS = ["launch the product next friday"]
    hyp_decisions = ["launch the product next friday", "hire two interns"]
    REF_ACTIONS = ["priya -> send the report -> friday"]
    hyp_actions = ["priya -> send the report -> friday"]

In [ ]:
nlp_results = {}

# ---- ASR ----
if REF_TRANSCRIPT:
    nlp_results["WER"] = word_error_rate(REF_TRANSCRIPT.lower(), hyp_transcript.lower())
    nlp_results["CER"] = character_error_rate(REF_TRANSCRIPT.lower(), hyp_transcript.lower())

# ---- Summarization ----
if REF_SUMMARY:
    try:
        nlp_results.update(evaluate_summary(hyp_summary, REF_SUMMARY))
    except ImportError as e:
        print("[WARN] install `rouge-score` and `bert-score` for summary metrics:", e)

# ---- Decisions / action items ----
if REF_DECISIONS:
    for k, v in precision_recall_f1(hyp_decisions, REF_DECISIONS).items():
        nlp_results[f"Decisions {k}"] = v
if REF_ACTIONS:
    for k, v in precision_recall_f1(hyp_actions, REF_ACTIONS).items():
        nlp_results[f"Actions {k}"] = v

nlp_df = pd.Series(nlp_results, name="score", dtype=object).to_frame()
nlp_df.to_csv(os.path.join(OUT_DIR, "nlp_asr_results.csv"))
nlp_df

Notes: WER/CER are lower-is-better; ROUGE, BERTScore and Precision/Recall/F1 are higher-is-better.
`precision_recall_f1` uses exact (normalized) string matching, which is strict for free-text action items —
for real evaluation, swap in an embedding-similarity threshold.

## 5. Export summary

In [ ]:
summary = {
    "settings": {"N_GEN": N_GEN, "DDIM_STEPS": DDIM_STEPS, "REFINE_T": REFINE_T, "device": str(DEVICE)},
    "fid_floor_real_vs_real": fid_floor,
    "representation_models": json.loads(results.to_json(orient="index")),
    "asr_nlp": {k: (None if v is None else float(v)) for k, v in nlp_results.items()},
}
with open(os.path.join(OUT_DIR, "evaluation_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Saved to", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))

### How to read the results
- **Best reconstruction** (highest PSNR/SSIM): usually the AE, since it optimizes only reconstruction. The VAE trades some fidelity for a smoother latent space; diffusion's score depends on `REFINE_T`.
- **Best generation** (lowest FID, diversity ratio near 1): this is where VAE, GAN and diffusion should separate. The AE's sampled-latent FID shows what you get without a trained prior.
- A model that is strong at both reconstruction and generation is a good choice for the representation-learning block of the pipeline.